# Step 2 — Turning Articles into a Machine-Generated Quiz

In Step 1 we built a corpus of fresh Guardian articles with full body text.
Now an LLM reads each article and writes **multiple-choice questions** about
it — the quiz that Steps 3–5 will judge, answer, and race on.

Along the way you'll learn four transferable skills:

1. **System vs. user messages** — how to split a prompt into a constant
   contract and a variable payload.
2. **Structured outputs** — forcing the model to return JSON that validates
   against a Pydantic schema, on *two different providers* (OpenAI & Gemini).
3. **Prompt design** — what separates a fair, well-grounded quiz question
   from a lazy one.
4. **`ThreadPoolExecutor`** — how to run many API calls in parallel with
   Python threads, and why that works despite the GIL.

## 1. Where we are

Load the Step 1 articles and look at one:

In [1]:
from toolkit.utils import load_jsonl

articles = load_jsonl("../data/articles/guardian_articles.jsonl")
article = articles[0]

print(f"{len(articles)} articles loaded")
print()
print("HEADLINE:", article["headline"])
print("SECTION :", article["section"], "|", article["published"])
print()
print(article["body_text"][:300] + " ...")

5 articles loaded

HEADLINE: Shabana Mahmood expected to be named Andy Burnham’s chancellor
SECTION : Politics | 2026-07-15T18:52:25Z

Shabana Mahmood has emerged as the frontrunner to become Andy Burnham’s chancellor after a fierce briefing war over the prospect of Ed Miliband being appointed to the powerful role. Senior Labour figures with knowledge of Burnham’s thinking told the Guardian they expected the home secretary to be mo ...


## 2. Loading API keys the portable way

You'll need two keys for this notebook (both have free tiers):

- **OpenAI** → https://platform.openai.com/api-keys → `export OPENAI_API_KEY=...`
- **Gemini** → https://aistudio.google.com/apikey → `export GEMINI_API_KEY=...`

In our lab, shared machines set keys with an **`SML_` prefix**
(`SML_OPENAI_API_KEY`) so personal and lab keys can coexist. The toolkit's
`load_api_key()` checks the prefixed name first, then falls back to the
standard one — so the *same code* runs on a lab machine and on your laptop.

Two design details worth copying in your own projects:

- it **logs** which variable it used (transparency) instead of `print`ing —
  callers control verbosity via the logging config;
- it raises **`ValueError`** when no key is found: a configuration mistake by
  the caller, not a runtime fault.

In [2]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s — %(message)s")

from toolkit.providers import PROVIDER_ENV, load_api_key

print("Canonical env var names:", PROVIDER_ENV)
key = load_api_key(PROVIDER_ENV["openai"])   # watch the log line
print("Got a key of length", len(key))

INFO toolkit.providers._keys — Loaded API key from environment variable: SML_OPENAI_API_KEY


Canonical env var names: {'openai': 'OPENAI_API_KEY', 'gemini': 'GEMINI_API_KEY', 'anthropic': 'ANTHROPIC_API_KEY', 'perplexity': 'PERPLEXITY_API_KEY', 'xai': 'XAI_API_KEY'}
Got a key of length 164


## 3. One raw OpenAI call — system + user messages

Every LLM request we make in this tutorial has **two messages**:

| Message | Contains | Changes per call? |
|---|---|---|
| **system** (OpenAI calls it `developer`) | the model's role + the rules of the job | no — it's the constant contract |
| **user** | the actual payload: this article, this request | yes — templated from data |

Why split? The system message is where behavioral instructions carry the most
weight, and keeping the payload separate means you can swap articles without
touching the rules — or improve the rules without touching the pipeline.

We also pass `text_format=` — a **Pydantic class** describing the JSON we
demand back. More on that in section 5; first, see it work:

In [3]:
from openai import OpenAI
from toolkit import prompts
from toolkit.questions import ArticleQuestions

client = OpenAI(api_key=load_api_key(PROVIDER_ENV["openai"]))

response = client.responses.parse(
    model="gpt-5.6-terra",
    input=[
        {"role": "developer", "content": prompts.MCQ_SYSTEM_PROMPT},
        {"role": "user", "content": prompts.build_mcq_user_prompt(
            article["headline"], article["body_text"], n_questions=3)},
    ],
    text_format=ArticleQuestions,
)

quiz = response.output_parsed        # <- a validated ArticleQuestions object
q = quiz.questions[0]
print(q.question)
for letter, option in zip("ABCD", q.options):
    mark = "*" if letter == q.correct_letter else " "
    print(f"  {mark}{letter}. {option}")
print()
print("WHY:", q.explanation)

INFO toolkit.providers._keys — Loaded API key from environment variable: SML_OPENAI_API_KEY


INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


After reports that Shabana Mahmood was the favourite for No 11, by how many percentage points did the yield on a 10-year UK government bond drop during Wednesday?
   A. 0.02 percentage points
   B. 0.04 percentage points
  *C. 0.06 percentage points
   D. 0.10 percentage points

WHY: The article states that “the yield on a 10-year bond” dropped “0.06 percentage points during the day.”


## 4. What makes a GOOD quiz question?

Look inside `toolkit/toolkit/prompts.py` — the system prompt is a checklist
of quiz-writing craft. Every rule exists because Steps 3–5 depend on it:

| Rule in `MCQ_SYSTEM_PROMPT` | Why it matters downstream |
|---|---|
| answerable **only from the article** | Step 4's closed-book LLMs shouldn't get it right from training data |
| question **stands alone** (names, dates, no "according to the article") | Step 5's human contestants see the question *without* the article |
| exactly one defensible answer | Step 3's judge needs an unambiguous ground truth |
| distractors: same category & granularity | guessing stays at 25%; no free eliminations |
| no "All of the above", no longest-option or always-A tells | test-taking tricks shouldn't beat reading |
| explanation quotes the article | Step 3 can verify faithfulness mechanically |

Would the model do this on its own? Let's ask with a **lazy prompt** and
compare:

In [4]:
lazy = client.responses.parse(
    model="gpt-5.6-terra",
    input=[
        {"role": "developer", "content": "You write multiple-choice questions."},
        {"role": "user", "content": f"Write 3 multiple-choice questions about this article:\n\n{article['body_text']}"},
    ],
    text_format=ArticleQuestions,
)

print("LAZY PROMPT:")
print(" ", lazy.output_parsed.questions[0].question)
print()
print("CRAFTED PROMPT:")
print(" ", quiz.questions[0].question)

INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


LAZY PROMPT:
  Who was reported to be the frontrunner to become Andy Burnham’s chancellor?

CRAFTED PROMPT:
  After reports that Shabana Mahmood was the favourite for No 11, by how many percentage points did the yield on a 10-year UK government bond drop during Wednesday?


Typical differences you'll see: the lazy question leans on "the article" or
"according to the text" (useless without the article in front of you), picks
the easiest surface facts, and writes throwaway distractors. The crafted
prompt produces questions that work as *stand-alone trivia* — which is
exactly what a live quiz website needs.

## 5. Structured outputs: the Pydantic contract

Free-text LLM output is a parsing nightmare ("Sure! Here are your
questions..."). Instead, both providers accept a **schema** and constrain the
model to emit JSON matching it. We define the schema once, in
`toolkit/toolkit/questions.py`:

```python
class MCQuestion(BaseModel):
    question: str            = Field(description="A self-contained question ...")
    options: list[str]       = Field(description="Exactly 4 answer options ...",
                                     min_length=4, max_length=4)
    correct_letter: Literal["A", "B", "C", "D"] = Field(description="...")
    explanation: str         = Field(description="1-3 sentences quoting ...")

class ArticleQuestions(BaseModel):
    questions: list[MCQuestion]
```

Three things to notice:

- **`Literal["A","B","C","D"]`** becomes a JSON-schema *enum* — the model
  physically cannot return "E" or "b".
- **`Field(description=...)`** strings are sent to the model as part of the
  schema — free extra prompting, right next to the field it describes.
- The SDK returns a **validated Python object**, so every record Steps 3–5
  read from disk is guaranteed to have 4 options and a legal answer letter.

In [5]:
# The schema both SDKs receive (excerpt):
import json
schema = ArticleQuestions.model_json_schema()
print(json.dumps(schema["$defs"]["MCQuestion"]["properties"]["correct_letter"], indent=2))

{
  "description": "The letter of the one correct option.",
  "enum": [
    "A",
    "B",
    "C",
    "D"
  ],
  "title": "Correct Letter",
  "type": "string"
}


## 6. The same call on Gemini

Different company, different SDK (`google-genai`), different parameter names —
**same ideas**. The system prompt becomes `system_instruction`; the Pydantic
class goes in `response_schema`; the parsed object comes back on
`response.parsed`:

In [6]:
from google import genai
from google.genai import types

gclient = genai.Client(api_key=load_api_key(PROVIDER_ENV["gemini"]))

gresponse = gclient.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompts.build_mcq_user_prompt(
        article["headline"], article["body_text"], n_questions=3),
    config=types.GenerateContentConfig(
        system_instruction=prompts.MCQ_SYSTEM_PROMPT,
        response_mime_type="application/json",
        response_schema=ArticleQuestions,
    ),
)

gq = gresponse.parsed.questions[0]      # the SAME Pydantic type as before!
print(type(gresponse.parsed).__name__, "—", gq.question)

INFO toolkit.providers._keys — Loaded API key from environment variable: SML_GEMINI_API_KEY


WARNING google_genai._api_client — Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


INFO google_genai.models — AFC is enabled with max remote calls: 10.


INFO httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


ArticleQuestions — Why did some Labour MPs warn against appointing Ed Miliband as chancellor?


That's the punchline of the provider abstraction: **two SDK dialects, one
schema, one prompt**. The toolkit wraps each dialect in a module with an
identical signature —

```python
toolkit.providers.openai_provider.run_parsed(model, system_prompt, user_text, response_format)
toolkit.providers.gemini_provider.run_parsed(model, system_prompt, user_text, response_format)
```

— so everything downstream treats "the provider" as an interchangeable
function. (Both also add tenacity retries for transient 429/5xx errors, just
like Step 1's Guardian client.)

## 7. Records, JSONL, and resume — Step 1 déjà vu

We flatten each validated response into **one JSONL line per question**:

```json
{"id": "openai__politics/2026/jul/15/some-slug__q0",
 "article_id": "politics/2026/jul/15/some-slug",
 "question_index": 0, "provider": "openai", "model": "gpt-5.6-terra",
 "question": "...", "options": ["...","...","...","..."],
 "correct_letter": "B", "explanation": "...", "generated_at": "..."}
```

One-line-per-question is what every later step wants to consume (the judge
scores a question; a contestant answers a question). The `provider` field
lets us keep OpenAI's and Gemini's quizzes apart, and the composite `id`
stays unique even if the files are concatenated.

Persistence is exactly Step 1's playbook — **append after every article**
(a crash costs one article, not the run) and **resume** by reading the ids
already saved, except this time skipping an article saves an *LLM call*
(money), not just an API call (quota). See `append_records()` and
`load_completed_article_ids()` in `toolkit/toolkit/questions.py`.

## 8. A sequential loop, timed

Generate questions for several articles the obvious way — one at a time —
and time it. `run_parsed` here is the toolkit's OpenAI adapter:

In [7]:
import time
from toolkit.providers import openai_provider

demo_articles = articles[:4]

def make_quiz(article):
    """One article -> one LLM call -> validated ArticleQuestions."""
    parsed, _raw = openai_provider.run_parsed(
        "gpt-5.6-terra",
        prompts.MCQ_SYSTEM_PROMPT,
        prompts.build_mcq_user_prompt(article["headline"], article["body_text"], 3),
        ArticleQuestions,
    )
    return parsed

t0 = time.monotonic()
for a in demo_articles:
    start = time.monotonic()
    quiz = make_quiz(a)
    print(f"  {time.monotonic()-start:5.1f}s  {a['headline'][:60]}")
sequential_seconds = time.monotonic() - t0
print(f"\nSequential total: {sequential_seconds:.1f}s for {len(demo_articles)} articles")

INFO toolkit.providers._keys — Loaded API key from environment variable: SML_OPENAI_API_KEY


INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


    4.1s  Shabana Mahmood expected to be named Andy Burnham’s chancell


INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


    3.2s  Andy Burnham signals a wealth tax is off the agenda for now


INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


    3.9s  Save the Children clashes with Labour after accusing Starmer


INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


    3.9s  At his final PMQs, Starmer soaked up the love from all sides

Sequential total: 15.1s for 4 articles


## 9. `ThreadPoolExecutor`: parallel I/O for free

Each call above spent almost all of its time **waiting** — the request
travels to a data center, a model thinks, the response travels back. Your
CPU sat idle.

**Why threads help here (despite the GIL):** Python's Global Interpreter
Lock means only one thread executes *Python bytecode* at a time — threads
won't speed up CPU-bound math. But a thread that's waiting on the network
**releases the GIL**, so eight threads can all be waiting at once. API calls
are the textbook I/O-bound workload; the waits overlap and the wall clock
collapses.

The pattern, from `concurrent.futures`:

- `pool.submit(fn, arg)` schedules a call and immediately returns a
  **`Future`** — a receipt you can redeem later;
- `as_completed(futures)` yields each future **as it finishes** — not in
  submission order — so you process results the moment they're ready.

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed

t0 = time.monotonic()
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(make_quiz, a): a for a in demo_articles}
    for future in as_completed(futures):          # finish order, not submit order!
        a = futures[future]
        quiz = future.result()
        print(f"  {time.monotonic()-t0:5.1f}s  done: {a['headline'][:60]}")
parallel_seconds = time.monotonic() - t0

print(f"\nParallel total:   {parallel_seconds:.1f}s  "
      f"(sequential was {sequential_seconds:.1f}s -> "
      f"{sequential_seconds/parallel_seconds:.1f}x speedup)")

INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


    3.5s  done: Save the Children clashes with Labour after accusing Starmer
    3.6s  done: Andy Burnham signals a wealth tax is off the agenda for now


INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


    3.9s  done: At his final PMQs, Starmer soaked up the love from all sides
    4.0s  done: Shabana Mahmood expected to be named Andy Burnham’s chancell

Parallel total:   4.0s  (sequential was 15.1s -> 3.8x speedup)


### What happens when one call fails?

`future.result()` re-raises any exception the worker hit. Uncaught, one bad
article would kill the whole run — so we catch **per future**. Watch, with a
fake provider so it costs nothing:

In [9]:
def flaky_make_quiz(article):
    if article["id"] == "fake/2":
        raise RuntimeError("simulated: provider returned garbage")
    time.sleep(0.1)   # pretend network wait
    return f"quiz for {article['id']}"

fake_articles = [{"id": f"fake/{i}"} for i in range(5)]

with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(flaky_make_quiz, a): a for a in fake_articles}
    for future in as_completed(futures):
        a = futures[future]
        try:
            print("  ok  :", future.result())
        except Exception as e:
            print("  FAIL:", a["id"], "—", e)

print("\nThe run survived; only fake/2 is missing (resume retries it later).")

  FAIL: fake/2 — simulated: provider returned garbage


  ok  : quiz for fake/0
  ok  : quiz for fake/4
  ok  : quiz for fake/1
  ok  : quiz for fake/3

The run survived; only fake/2 is missing (resume retries it later).


### Two more thread rules we follow

1. **Only the main thread writes the file.** Workers just call the API and
   return records; the `as_completed` loop appends them. That's why the
   toolkit needs no locks. (The alternative — many threads sharing a file
   handle guarded by `threading.Lock` — works too, but there's simply nothing
   to reason about when one writer owns the file.)
2. **Share one client.** Both SDK clients are thread-safe, so all workers use
   the same connection pool instead of paying per-thread setup.

And the honest caveat: **more workers ≠ more speed forever.** Past the
provider's rate limit you just collect 429s; the tenacity backoff inside each
worker absorbs them, but you stop gaining. Start at 8 and measure.

## 10. The payoff: it all lives in `toolkit.questions`

Prompts, schema, provider adapters, threadpool, JSONL, resume — packaged:

| Notebook section | Toolkit home |
|---|---|
| §2 key loading | `toolkit.providers.load_api_key()` |
| §3–4 prompts | `toolkit.prompts.MCQ_SYSTEM_PROMPT`, `build_mcq_user_prompt()` |
| §5 schema | `toolkit.questions.MCQuestion`, `ArticleQuestions` |
| §3 / §6 provider calls | `toolkit.providers.{openai,gemini}_provider.run_parsed()` |
| §7 records + resume | `to_question_records()`, `load_completed_article_ids()` |
| §8–9 loop / threadpool | **`generate_for_articles(..., parallel=True)`** |

In [10]:
from toolkit.questions import generate_for_articles

summary = generate_for_articles(
    articles[:4],
    output_fp="../data/questions/demo_questions.jsonl",
    provider="openai",
    parallel=True,
    max_workers=4,
)
summary

INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO toolkit.questions — [1/4] politics/2026/jul/15/andy-burnham-wealth-tax-off-agenda-for-now -> 3 questions


INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO toolkit.questions — [2/4] uk-news/2026/jul/15/save-the-children-accusing-starmer-of-complicity-in-gaza-deaths -> 3 questions


INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO toolkit.questions — [3/4] politics/2026/jul/15/shabana-mahmood-andy-burnham-chancellor -> 3 questions


INFO httpx — HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO toolkit.questions — [4/4] politics/2026/jul/15/at-his-final-pmqs-starmer-soaked-up-the-love-from-all-sides-and-even-some-tenderness-from-kemi -> 3 questions


{'provider': 'openai',
 'model': 'gpt-5.6-terra',
 'articles_processed': 4,
 'articles_skipped': 0,
 'articles_failed': 0,
 'new_questions': 12,
 'elapsed_seconds': 5.26,
 'output_fp': '../data/questions/demo_questions.jsonl'}

In [11]:
# Run the exact same cell again — resume skips every article, zero API calls:
generate_for_articles(
    articles[:4],
    output_fp="../data/questions/demo_questions.jsonl",
    provider="openai",
    parallel=True,
)

INFO toolkit.questions — Resuming: skipping 4 articles already in ../data/questions/demo_questions.jsonl


{'provider': 'openai',
 'model': 'gpt-5.6-terra',
 'articles_processed': 0,
 'articles_skipped': 4,
 'articles_failed': 0,
 'new_questions': 0,
 'elapsed_seconds': 0.0,
 'output_fp': '../data/questions/demo_questions.jsonl'}

## 11. The research-grade CLI

[`scripts/02_generate_questions.py`](../scripts/02_generate_questions.py)
wraps `generate_for_articles` in the same argparse shell as Step 1. The
`--parallel` flag switches from the sequential loop to the threadpool:

In [12]:
!cd .. && uv run python scripts/02_generate_questions.py --help

usage: 02_generate_questions.py [-h] [--input INPUT]
                                --model {gemini-3.1-flash-lite,gemini-3.5-flash,gpt-5.4-mini-2026-03-17,gpt-5.6-terra}
                                [--n-questions N_QUESTIONS]
                                [--max-articles MAX_ARTICLES] [--parallel]
                                [--max-workers MAX_WORKERS] [--output OUTPUT]
                                [--no-resume]
                                [--log-level {DEBUG,INFO,WARNING,ERROR}]
                                [--log-file LOG_FILE | --create-log-file]

Purpose: Generate multiple-choice questions from Step 1's Guardian articles
using an LLM (OpenAI or Gemini) with structured outputs, optionally in
parallel via a thread pool.

Examples
--------
# 3 questions per article for every Step-1 article, with an OpenAI model
uv run python scripts/02_generate_questions.py --model gpt-5.6-terra

# A Gemini question set (run several models to enable Step 3+ comparisons)
uv run py

The two-run recipe that sets up the rest of the tutorial:

```bash
uv run python scripts/02_generate_questions.py --model gpt-5.6-terra --parallel
uv run python scripts/02_generate_questions.py --model gemini-3.1-flash-lite --parallel
```

→ `data/questions/questions_gpt-5.6-terra.jsonl` + `questions_gemini-3.1-flash-lite.jsonl`: two
complete quizzes over the same articles, by two different model families.

---

### Next up: Step 3 ⚖️

Machine-written questions are cheap — but are they *good*? In the next
notebook an **LLM judge** cross-examines every question for quality,
faithfulness to its article, and guessability, and only the survivors go into
the horse race.